# Tools

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("what is rag")
response.content

'<think>\nOkay, so I need to explain what RAG is. Hmm, RAG... I remember it\'s something related to artificial intelligence, maybe in the context of large language models. Let me think. I think it stands for Retrieval-Augmented Generation. Right, that sounds familiar. So, the basic idea is that instead of just relying on the model\'s training data, it can retrieve information from external sources when answering a query.\n\nWait, how does that work exactly? So, when a user asks a question, the model doesn\'t just generate an answer based on its internal knowledge. It first looks up relevant information from a database or some other external source, and then uses that information to generate a more accurate or up-to-date answer. That makes sense because even though large language models are trained on a lot of data, they might not have the latest information or specific details that are stored in databases.\n\nLet me break it down. The retrieval part would involve querying a database or

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """get the weather at a given location"""
    return f"its cold in {location}"

model_with_tools=model.bind_tools([get_weather])

In [ ]:
resoponse = model_with_tools.invoke("what is weather in germany?")
print(resoponse)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


# Tool Execution Loops

In [ ]:
# step 1: Model generates tools calls
messages = [{"role":"user", "content":"what is the weather in germany"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


The weather in Germany is currently cold. You might want to dress warmly if you're visiting or living there! ❄️


In [ ]:
messages

[{'role': 'user', 'content': 'what is the weather in germany'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Germany. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Germany is the location here. I need to call that function with "Germany" as the argument. Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': 'y5qy0kgbm', 'function': {'arguments': '{"location":"Germany"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 153, 'total_tokens': 242, 'completion_time': 0.141199594, 'completion_tokens_details': {'reasoning_tokens': 65}, 'prompt_time': 0.007958185, 'prompt_tokens_details': None, 'queue_time': 0.160512574, 'total_time': 0.149157779}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_r

# Built-in Tool - DuckDuckGO Search

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun
search = DuckDuckGoSearchRun()

result = search.invoke("tech stack of fde engineer?")
print(result)

3 days ago - Forward Deployed Engineer (FDE) is a professional role within information technology and software engineering in which an engineer works closely with a client organization to develop, customize, and deploy technical solutions in operational environments. The role combines software development ... 6 days ago - An FDE stack is customer-shaped rather than product-shaped — every tool either ingests customer context, runs an eval against customer data, or deploys into a customer environment. February 26, 2026 - But when they hit a technical wall — a missing API, a brittle migration, a product limitation — they need engineering. FDEs eliminate that dependency by solving the technical blockers themselves. ... CSEs, SEs, and Consultants operate around the product. FDEs operate inside it. ... While AI is making every role more “full-stack,” FDEs are full-stack in a completely different direction. February 11, 2026 - To become a successful forward-deployed engineer, you need both te

In [7]:
print(search.name)
print(search.description)
print(search.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


# Built-in Tool - Shell Tool

In [ ]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

result = shell_tool.invoke('dir')

print(result)

# Custom Tools

In [10]:
from langchain_core.tools import tool


In [16]:
# step 1 - create a function
def addition(x,y):
    """ adding two numbers"""
    return x+y

In [17]:
# step 2 - adding type hints
def addition(x: int,y: int) -> int:
    """ adding two numbers"""
    return x+y

In [18]:
# step 3 - adding tool decorator

@tool
def addition(x: int,y: int) -> int:
    """ adding two numbers"""
    return x+y

In [19]:
result = addition.invoke({"x":9, "y":7})


In [20]:
print(result)

16


In [21]:
print(addition.name)
print(addition.description)
print(addition.args)

addition
adding two numbers
{'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}


In [22]:

print(addition.args_schema.model_json_schema())

{'description': 'adding two numbers', 'properties': {'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}, 'required': ['x', 'y'], 'title': 'addition', 'type': 'object'}


# Method 2 - Using StruturedTool

In [26]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field


In [30]:
class AdditionInput(BaseModel):
    x: int =  Field(..., description=" frist number to be added")
    y: int =  Field(..., description="second number to added")

In [32]:
def addition_func(x: int, y: int) -> int:
    return x+y

In [34]:
addition_tool = StructuredTool.from_function(
    func= addition_func,
    name="addition",
    description="adding two given numbers",
    args_schema=AdditionInput
)

In [35]:
result = addition_tool.invoke({'x':4, 'y':8})
from distro import name

print(result)
print(addition_tool.name)
print(addition_tool.description)
print(addition_tool.args)

12
addition
adding two given numbers
{'x': {'description': ' frist number to be added', 'title': 'X', 'type': 'integer'}, 'y': {'description': 'second number to added', 'title': 'Y', 'type': 'integer'}}


# Method3 - Using BaseTool Class

In [36]:
from langchain.tools import BaseTool
from typing import Type

In [37]:
# arg schema using pydantic
class AdditionInput(BaseModel):
    x: int = Field(..., description="the frist number to add")
    y: int = Field(..., description="adding second number")

In [38]:
class AdditionTool(BaseTool):
    name: str = "addition"
    description: str = " adding two given numbers"

    args_schema: Type[BaseModel] = AdditionInput

    def _run(self, x: int, y: int) -> int:
        return x+y

In [39]:
addition_tool = AdditionTool()

In [41]:
result = addition_tool.invoke({'x':3, 'y':8})

print(result)
print(addition_tool.name)
print(addition_tool.description)
print(addition_tool.args)

11
addition
 adding two given numbers
{'x': {'description': 'the frist number to add', 'title': 'X', 'type': 'integer'}, 'y': {'description': 'adding second number', 'title': 'Y', 'type': 'integer'}}


# Toolkit

In [42]:
from langchain_core.tools import tool

# custom tolls
@tool
def add(x: int, y: int) -> int:
    """ adding two numbers"""
    return x+y

@tool 
def multiply(x: int, y: int) -> int:
    """ multiplying two numbers """
    return x*y    

In [43]:
class Mathtoolkit:
    def get_tools(self):
        return[add, multiply]

In [44]:
toolkit = Mathtoolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => adding two numbers
multiply => multiplying two numbers
